In [2]:
import numpy as np
import pandas as pd
from scipy import stats

In [3]:
deposits_size = pd.read_csv('../int/deposits_size.csv')
deposits_category = pd.read_csv('../int/deposits_category.csv')
staked_pool_size = pd.read_csv('../int/active_validators_size.csv')
staked_category = pd.read_csv('../int/active_validators_category.csv')

deposits_size = deposits_size[deposits_size['slot'] >= 6206400]
deposits_category = deposits_category[deposits_category['slot'] >= 6206400]

In [4]:
rewards_size = pd.read_csv('../int/rewards_size.csv', usecols=('1', '2-5', '6-19', '20-99', '100+', 'slot'))
rewards_category = pd.read_csv('../int/rewards_category.csv', usecols=('slot', 'CEX', 'Liquid Restaking', 'Liquid Staking', 'Solo Stakers', 'Staking Pools'))

columns_to_compare = ['1', '2-5', '20-99', '6-19']
reference_column = '100+'

# Calculate the percentage difference compared to '100+'
for col in columns_to_compare:
    rewards_size[f'{col}'] = ((rewards_size[col] - rewards_size[reference_column]) / rewards_size[reference_column])

# Display the updated DataFrame
rewards_size = rewards_size.drop(columns=('100+'))


# List of columns to calculate the percentage difference for
columns_to_compare = ['CEX', 'Liquid Restaking', 'Liquid Staking', 'Staking Pools']
reference_column = 'Solo Stakers'

# Calculate the percentage difference compared to 'Solo Stakers'
for col in columns_to_compare:
    rewards_category[f'{col}'] = ((rewards_category[col] - rewards_category[reference_column]) / rewards_category[reference_column])

# Display the updated DataFrame
rewards_category = rewards_category.drop(columns=('Solo Stakers'))

In [5]:
deposits_size_set = deposits_size
deposits_size_set['slot'] = deposits_size_set['slot'] // 7200 * 7200

# Group by the day and get the last slot of each day and sum values for all columns
deposits_size_set = deposits_size_set.groupby('slot').agg(
    {
        'slot': 'last',
        '1': 'sum',
        '2-5': 'sum',
        '6-19': 'sum',
        '20-99': 'sum',
        '100+': 'sum',
        'total': 'sum'
    }
).reset_index(drop=True)

deposits_size_set

,slot,1,2-5,6-19,20-99,100+,total
0,6206400,64.0,128.0,512.0,5184.0,14720.0,20608.0
1,6213600,192.0,768.0,640.0,3520.0,66800.0,71920.0
2,6220800,768.0,896.0,2240.0,11040.0,106816.0,121760.0
3,6228000,448.0,704.0,1920.0,4256.0,50320.0,57648.0
4,6235200,256.0,864.0,2720.0,5248.0,85296.0,94384.0
...,...,...,...,...,...,...,...
382,8956800,224.0,448.0,320.0,192.0,32268.0,33452.0
383,8964000,160.0,480.0,64.0,256.0,17014.0,17974.0
384,8971200,1248.0,224.0,960.0,4384.0,9730.0,16546.0
385,8978400,192.0,384.0,64.0,192.0,7203.0,8035.0


In [6]:
deposits_category_set = deposits_category
deposits_category_set['slot'] = deposits_category_set['slot'] // 7200 * 7200

# Create a dictionary for aggregation
agg_dict = {col: 'sum' for col in deposits_category_set.columns if col != 'slot'}
agg_dict['slot'] = 'last'

# Group by 'slot' and apply the aggregation
deposits_category_set = deposits_category_set.groupby('slot').agg(agg_dict).reset_index(drop=True)

deposits_category_set

,CEX,Liquid Restaking,Liquid Staking,Solo Stakers,Staking Pools,Unidentified,total,slot
0,4544.0,0.0,5600.0,224.0,576.0,9664.0,20608.0,6206400
1,9920.0,0.0,2928.0,192.0,2976.0,55904.0,71920.0,6213600
2,36608.0,0.0,28800.0,1888.0,3392.0,51072.0,121760.0,6220800
3,27616.0,0.0,13104.0,1344.0,2176.0,13408.0,57648.0,6228000
4,29984.0,0.0,13072.0,6976.0,1440.0,42912.0,94384.0,6235200
...,...,...,...,...,...,...,...,...
382,2592.0,23440.0,316.0,64.0,576.0,6464.0,33452.0,8956800
383,704.0,1200.0,902.0,32.0,8352.0,6784.0,17974.0,8964000
384,3488.0,768.0,226.0,0.0,1056.0,11008.0,16546.0,8971200
385,2048.0,352.0,1027.0,32.0,448.0,4128.0,8035.0,8978400


In [7]:
staked_pool_size_set = staked_pool_size[staked_pool_size['slot'].isin(deposits_size_set['slot'])]
staked_pool_size_set.reset_index(inplace=True)
staked_pool_size_set = staked_pool_size_set.drop(columns=['index'])

staked_category_set = staked_category[staked_category['slot'].isin(deposits_category_set['slot'])]
staked_category_set.reset_index(inplace=True)
staked_category_set = staked_category_set.drop(columns=['index'])

deposits_size_set.set_index('slot', inplace=True)
staked_pool_size_set.set_index('slot', inplace=True)
deposits_percentage_size = deposits_size_set.divide(staked_pool_size_set, fill_value=0) * 100
deposits_percentage_size.reset_index(inplace=True)

deposits_category_set.set_index('slot', inplace=True)
staked_category_set.set_index('slot', inplace=True)
deposits_percentage_category = deposits_category_set.divide(staked_category_set, fill_value=0) * 100
deposits_percentage_category.reset_index(inplace=True)

In [11]:
deposits_percentage_size = deposits_percentage_size.drop(columns=(['100+', 'total']))

In [12]:
import numpy as np
import pandas as pd
from scipy import stats

# Assuming the DataFrames are already defined and loaded
# deposits_percentage_size and rewards_size

# Calculate the percent change for the rewards_size DataFrame
rewards_size_pct_change = rewards_size.set_index('slot').pct_change().reset_index()

# Merge the DataFrames on the slot column for percent change calculations
price_elasticity_size = pd.merge(deposits_percentage_size, rewards_size_pct_change, on='slot')

# Replace infinite values with NaN and drop rows with NaN values for elasticity calculations
price_elasticity_size.replace([np.inf, -np.inf], np.nan, inplace=True)
price_elasticity_size.dropna(inplace=True)

# Merge the original DataFrames on the slot column for correlation calculations
merged_df = pd.merge(deposits_percentage_size, rewards_size, on='slot', suffixes=('_exit', '_apy'))

# Replace infinite values with NaN for correlation calculations
merged_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Initialize dictionaries to store the results
elasticity = {}
t_stats = {}
standard_deviations = {}
p_values = {}
counts = {}
correlations = {}

# List of columns to calculate elasticity and correlation for
columns = deposits_percentage_size.columns.drop(['slot'])
for col in columns:
    price_elasticity_size[f'elasticity_{col}'] = price_elasticity_size[f'{col}_x'] / price_elasticity_size[f'{col}_y']
    
    # Drop NaN values for the specific column for elasticity calculations
    valid_elasticity = price_elasticity_size[f'elasticity_{col}'].dropna()
    
    # Store the results for elasticity
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    counts[col] = valid_elasticity.count()
    
    # Perform a one-sample t-test against zero
    t_stat, p_value = stats.ttest_1samp(valid_elasticity, 0)
    t_stats[col] = t_stat
    p_values[col] = p_value
    
    # Calculate the correlation coefficient using non-percent change values
    correlation, _ = stats.pearsonr(merged_df[f'{col}_exit'], merged_df[f'{col}_apy'])
    correlations[col] = correlation

# Print the results
print("Elasticity and Correlation Analysis Results:")
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t-stat = {t_stats[col]}, Std Dev = {standard_deviations[col]}, P-value = {p_values[col]}, Count = {counts[col]}, Correlation Coefficient = {correlations[col]}')

# Display the DataFrame with elasticity columns (optional)
print(price_elasticity_size)

Elasticity and Correlation Analysis Results:
1: Mean Elasticity = 69.28435634842371, t-stat = 1.2874694193797622, Std Dev = 928.9800691676429, P-value = 0.19893307607191046, Count = 298, Correlation Coefficient = -0.08717029579782419
2-5: Mean Elasticity = 2.9615984073227555, t-stat = 0.7892219025894895, Std Dev = 64.77913887794624, P-value = 0.4306118974007648, Count = 298, Correlation Coefficient = -0.04720881923091088
20-99: Mean Elasticity = -5.98170169853597, t-stat = -0.7960935561472082, Std Dev = 129.708600897148, P-value = 0.42661350252617514, Count = 298, Correlation Coefficient = 0.03763019552630401
6-19: Mean Elasticity = -14.038975041195252, t-stat = -1.082721708155819, Std Dev = 223.83432670194762, P-value = 0.27981021051317556, Count = 298, Correlation Coefficient = 0.08540539597067538
        slot        1_x     2-5_x    20-99_x     6-19_x       1_y     2-5_y  \
1    6847200   5.765766  4.812587   1.476404   3.230913  0.106203 -0.026545   
2    6854400   6.248123  1.1105

In [9]:
import numpy as np
import pandas as pd
from scipy import stats

# Assuming the DataFrames are already defined and loaded
# deposits_percentage_category and rewards_category

# Calculate the percent change for the rewards_category DataFrame
rewards_category_pct_change = rewards_category.set_index('slot').pct_change().reset_index()

# Merge the DataFrames on the slot column for percent change calculations (elasticity)
elasticity_df = pd.merge(deposits_percentage_category[['slot', 'Solo Stakers']], rewards_category_pct_change, on='slot')

# Merge the DataFrames on the slot column for original values (correlation)
correlation_df = pd.merge(deposits_percentage_category[['slot', 'Solo Stakers']], rewards_category, on='slot')

# Replace infinite values with NaN and drop rows with NaN values for elasticity calculations
elasticity_df.replace([np.inf, -np.inf], np.nan, inplace=True)
elasticity_df.dropna(inplace=True)

# Replace infinite values with NaN for correlation calculations
correlation_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Initialize dictionaries to store the results
elasticity = {}
t_stats = {}
standard_deviations = {}
p_values = {}
counts = {}
correlations = {}

# List of columns to calculate elasticity and correlation for
columns = ['CEX', 'Liquid Restaking', 'Liquid Staking', 'Staking Pools']
reference_column = 'Solo Stakers'

# Calculate elasticity and correlation for each column
for col in columns:
    # Calculate elasticity using percent change values
    elasticity_df[f'elasticity_{col}'] = elasticity_df[reference_column] / elasticity_df[f'{col}']
    
    # Drop NaN values for the specific column for elasticity calculations
    valid_elasticity = elasticity_df[[f'elasticity_{col}', reference_column, f'{col}']].dropna()
    
    # Store the results for elasticity
    elasticity[col] = valid_elasticity[f'elasticity_{col}'].mean()
    standard_deviations[col] = valid_elasticity[f'elasticity_{col}'].std()
    counts[col] = valid_elasticity[f'elasticity_{col}'].count()
    
    # Perform a one-sample t-test against zero
    t_stat, p_value = stats.ttest_1samp(valid_elasticity[f'elasticity_{col}'], 0)
    t_stats[col] = t_stat
    p_values[col] = p_value
    
    # Calculate the correlation coefficient using non-percent change values
    valid_correlation = correlation_df[[reference_column, col]].dropna()
    correlation, _ = stats.pearsonr(valid_correlation[reference_column], valid_correlation[col])
    correlations[col] = correlation

# Print the results
print("Elasticity and Correlation Analysis Results:")
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t-stat = {t_stats[col]}, Std Dev = {standard_deviations[col]}, P-value = {p_values[col]}, Count = {counts[col]}, Correlation Coefficient = {correlations[col]}')

# Display the DataFrame with elasticity columns (optional)
print(elasticity_df)
print(correlation_df)

Elasticity and Correlation Analysis Results:
CEX: Mean Elasticity = -0.9661007160207249, t-stat = -1.1588228905121898, Std Dev = 14.391745507624186, P-value = 0.24745927004223556, Count = 298, Correlation Coefficient = 0.03898442546330162
Liquid Restaking: Mean Elasticity = -0.927662254875507, t-stat = -2.2731802931375835, Std Dev = 7.044726481676038, P-value = 0.02372934774720833, Count = 298, Correlation Coefficient = -0.003693568685431528
Liquid Staking: Mean Elasticity = -0.9153316639496675, t-stat = -1.8691200552797202, Std Dev = 8.453750395449596, P-value = 0.06259017320340972, Count = 298, Correlation Coefficient = 0.05453544406536452
Staking Pools: Mean Elasticity = -1.2740053953416859, t-stat = -2.345509282985642, Std Dev = 9.376532065190732, P-value = 0.01965896769335531, Count = 298, Correlation Coefficient = 0.04057174666353967
        slot  Solo Stakers        CEX  Liquid Restaking  Liquid Staking  \
1    6847200      0.850258  -0.922164         -1.522252       -0.490931  